# Model SP accessibility vs. activity/travel patterns

In [1]:
%load_ext autoreload
%autoreload 2
%cd D:\netmob25

D:\netmob25


In [2]:
import os
os.environ['USE_PYGEOS'] = '0'
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
import h3.api.numpy_int as h3
from shapely.geometry import Polygon

# --- helper: weighted median ---
def weighted_median(values, weights):
    """
    Compute weighted median.
    values, weights -> 1D arrays of same length
    """
    sorter = np.argsort(values)
    values, weights = values[sorter], weights[sorter]
    cumsum = np.cumsum(weights)
    cutoff = weights.sum() / 2.0
    return values[np.searchsorted(cumsum, cutoff)]


In [3]:
import matplotlib as mpl
def setup_mpl():
    mpl.rc('font', size = 9)
    mpl.rcParams['legend.fontsize'] = 'small'
    mpl.rcParams['legend.fontsize'] = 'small'
    mpl.rcParams['xtick.labelsize'] = 'small'
    mpl.rcParams['ytick.labelsize'] = 'small'
    mpl.rcParams['lines.linewidth'] = 1
    mpl.rcParams['xtick.major.width'] = 1
    mpl.rcParams['ytick.major.width'] = 1
    mpl.rcParams['xtick.minor.width'] = 1
    mpl.rcParams['ytick.minor.width'] = 1
    mpl.rcParams['xtick.major.size'] = 3
    mpl.rcParams['ytick.major.size'] = 3
    mpl.rcParams['xtick.minor.size'] = 1.5
    mpl.rcParams['ytick.minor.size'] = 1.5
    mpl.rcParams['axes.linewidth'] = 1
    mpl.rcParams['xtick.top'] = False
    mpl.rcParams['ytick.right'] = False
    mpl.rcParams['mathtext.default'] = 'regular'
    mpl.rcParams['xtick.major.pad'] = '2.3'
    mpl.rcParams['ytick.major.pad' ]= '2.3'
    mpl.rcParams['axes.labelpad'] = 2

setup_mpl()

In [4]:
df = pd.read_csv("dbs/data_p/commuter_model_features_r.csv")

In [5]:
df.loc[:, 'ah_log'] = df['access_h'].apply(lambda x: x if x == 0 else np.log(x))
df.loc[:, 'ak_zero'] = df['ak_log'].apply(lambda x: 1 if x == 0 else 0)
df.columns

Index(['ID', 'entropy_mm', 'hill_q1', 'activity_nh_ratio',
       'activity_time_third', 'total_travel_time', 'xs_total_hws',
       'trip_chaining_presence', 'Gender', 'main_mode_r', 'Age', 'Education',
       'Household_type', 'weight_ind', 'access_h', 'mode', 'ak', 'ak_cat',
       'ak_log', 'codgeo', 'pt_sub', 'active_mode', 'h3_id', 'poverty_rate',
       'ah_log', 'ak_zero'],
      dtype='object')

## 0. VIF test on controls

In [6]:
def compute_vif_strict(df, features, one_hot=True, drop_first=True):
    X = df[features].copy()

    # 1) One-hot encode categoricals
    if one_hot:
        X = pd.get_dummies(X, drop_first=drop_first)

    # 2) Force numeric and remove bad rows/cols
    X = X.apply(pd.to_numeric, errors='coerce')
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X.dropna(axis=0, inplace=True)                 # drop rows with NaN
    # drop columns that are all-NaN or constant (zero variance)
    const_cols = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
    if const_cols:
        X.drop(columns=const_cols, inplace=True)

    # 3) Ensure pure float64 ndarray
    X_mat = X.to_numpy(dtype=float)
    cols  = X.columns.tolist()

    # 4) VIF for each column
    vif_vals = [variance_inflation_factor(X_mat, i) for i in range(X_mat.shape[1])]
    return pd.DataFrame({"feature": cols, "VIF": vif_vals}).sort_values("VIF", ascending=False), X

In [7]:
vars_to_check = ['Gender', 'mode', 'Household_type', 'pt_sub', 'ak_log', 'active_mode', 'Education', 'poverty_rate'] # , 'ah_log', 'Age', 'Education'
# ---- Example usage ----
vif_df, X_clean = compute_vif_strict(df.drop_duplicates(subset='ID'), vars_to_check)
print(vif_df)

               feature       VIF
7  mode_Public transit  5.761293
4            Education  4.673824
5         poverty_rate  3.884065
1               pt_sub  3.673272
3          active_mode  2.960166
0       Household_type  2.171827
6         Gender_Woman  2.098238
2               ak_log  1.530708


## 1. Descriptive statistics

In [7]:
vars2report = ['hill_q1', 'total_travel_time', 
               'Gender', 'main_mode_r', 'Age', 'Education', 'Household_type', 
               'weight_ind', 'mode', 'ak_cat', 'ak_log', 'pt_sub', 'active_mode', 'poverty_rate']
df2report = df.drop_duplicates(subset='ID').copy()

In [8]:
cat_dict = {'hill_q1': 'Leisure activity participation', 
            #'activity_time_third': 'Leisure activity participation', 
            'total_travel_time': "Trip making", 
            #'xs_total_hws': "Trip making", 
            #'trip_chaining_presence': "Trip making", 
            'Gender': "Individual attributes", 
            #'main_mode_r': "Transport mode", 
            'Age': "Individual attributes", 
            'Education': "Individual attributes", 
            'Household_type': "Individual attributes", 
            'weight_ind': "Individual attributes", 
            'mode': "Transport mode", 
            'ak_cat': "Space-time accessibility", 
            'ak_log': "Space-time accessibility", 
            'pt_sub': "Transport mode", 
            'active_mode': "Individual attributes",
            'poverty_rate': "Individual attributes"}
cat_name_dict = {'hill_q1': 'Leisure activity diversity', 
            #'activity_time_third': 'Third activity time (min)', 
            'total_travel_time': "Total travel time (min)", 
            #'xs_total_hws': "Trip chaining complexity", 
            #'trip_chaining_presence': "Trip chaining presence", 
            'Gender': "Gender", 
            # 'main_mode_r': "Main transport mode", 
            'Age': "Age", 
            'Education': "Education", 
            'Household_type': "Household type", 
            'weight_ind': "Individual attributes", 
            'mode': "Main transport mode", 
            'poverty_rate': "Poverty rate (IRIS zone level)",
            'ak_cat': "Space-time accessibility (> 0)", 
            'ak_log': "Space-time accessibility (log)", 
            'pt_sub': "Public transit subscription", 
            'active_mode': "Use of active mode"}
cat_list = ['Gender', 'Education', 'Household_type', 'mode', 'ak_cat', 'pt_sub', 'active_mode'] # 'trip_chaining_presence', 

In [9]:
df['active_mode'].unique()

array([1., 0.])

In [10]:
cat_level_dict = dict()
Education_dict = {
    'No diploma': 0,
    'Vocational': 1,
    'Lower secondary': 2,
    'Upper secondary': 3,
    '3–4-year higher education': 4,
    "5-year-and-above higher education": 5,
    "Missing": 9
}
household_order = {
    'Living alone': 0,
    'In a couple w/o children': 1,
    'Single parent': 2,
    'Living with parent(s)': 3,
    'Not related to other household members': 4,
    'In a shared apartment': 5,
    'In a couple w/ child(ren)': 6,
    'Another family member in the household': 7
}
cat_level_dict['Education'] = {v: k for k, v in Education_dict.items()}
# cat_level_dict['trip_chaining_presence'] = {0: 'w/o', 1: 'w/'}
cat_level_dict['Gender'] = {'Woman': 'Woman', 'Man': 'Man'}
cat_level_dict['Household_type'] = {v: k for k, v in household_order.items()}
cat_level_dict['mode'] = {'Car': 'Car', 'Public transit': 'Public transit'}
cat_level_dict['ak_cat'] = {'Zero': 'Zero', 'Non-zero': 'Non-zero'}
cat_level_dict['pt_sub'] = {True: 'Yes', False: 'No'}
cat_level_dict['active_mode'] = {1: 'Yes', 0: 'No'}

In [11]:
# --- Weighted mean and weighted SD helpers ---
def weighted_mean(x, w):
    return np.sum(x * w) / np.sum(w)

def weighted_sd(x, w):
    mean = weighted_mean(x, w)
    return np.sqrt(np.sum(w * (x - mean) ** 2) / np.sum(w))

# --- Separate continuous vs categorical ---
cont_vars = [col for col in cat_dict.keys() if col not in cat_list]
cat_vars  = cat_list
w = "weight_ind"   # column name for weights

summary_cont = []
for col in cont_vars:
    x = df2report[col].dropna()
    wts = df2report.loc[x.index, w]
    summary_cont.append({
        "variable": col,
        "category": None,
        "mean": weighted_mean(x, wts),
        "sd": weighted_sd(x, wts)
    })
summary_cont = pd.DataFrame(summary_cont)
summary_cont.loc[:, 'Name'] = summary_cont.loc[:, 'variable'].map(cat_name_dict)
summary_cont.loc[:, 'Group'] = summary_cont.loc[:, 'variable'].map(cat_dict)
# Create formatted column
summary_cont["mean_sd"] = summary_cont.apply(
    lambda row: f"{row['mean']:.1f} ({row['sd']:.1f})", axis=1
)

# --- Weighted shares for categorical variables ---
summary_cat = []
for col in cat_vars:
    # weighted share for each category
    counts = (
        df2report.groupby(col, dropna=False)[w]
        .apply(lambda x: x.sum() / df2report[w].sum()*100)
    )
    for k, v in counts.items():
        summary_cat.append({
            "variable": col,
            "category": k,
            "share": v
        })

summary_cat = pd.DataFrame(summary_cat)
summary_cat.loc[:, 'Name'] = summary_cat.loc[:, 'variable'].map(cat_name_dict)
summary_cat.loc[:, 'Group'] = summary_cat.loc[:, 'variable'].map(cat_dict)
summary_cat.loc[:, 'Level'] = summary_cat.apply(lambda row: cat_level_dict[row['variable']][row['category']], axis=1)

# --- Combine results ---
summary = {
    "continuous": summary_cont,
    "categorical": summary_cat
}

In [12]:
summary["continuous"].sort_values(by=['Group', 'Name']).to_clipboard(index=False)

In [14]:
summary["categorical"].sort_values(by=['Group', 'Name']).to_clipboard(index=False)

## 2. Statistics used in discussion
### 2.1 Household structure (2,6 with children)

In [ ]:
df_trips = pd.read_parquet(f'dbs/data_p/stays_extraction_all.parquet')
df_trips.head()

In [ ]:
# --- filter trips where origin OR destination is ACCOM ---
df_accom = df_trips[(df_trips["purpose_o"] == "ACCOM") | (df_trips["purpose_d"] == "ACCOM")].copy()

# --- group by ID and compute weighted median duration ---
wm_by_id = (
    df_accom.groupby("ID")
    .apply(lambda g: weighted_median(g["duration"].to_numpy(), g["weight_day"].to_numpy()))
    .reset_index(name="accom_time")
)

print(wm_by_id.head())

In [19]:
df = pd.merge(df, wm_by_id, on='ID', how='left')
df.fillna(0, inplace=True)

In [20]:
print(np.average(df["activity_nh_ratio"], weights=df["weight_ind"]))
df.groupby('Household_type').apply(lambda g: np.average(g["activity_nh_ratio"], weights=g["weight_ind"])).reset_index(name='activity_nh_ratio')

4.625795052964561


C:\Users\yuanlia\AppData\Local\Temp\ipykernel_9352\4145878640.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('Household_type').apply(lambda g: np.average(g["activity_nh_ratio"], weights=g["weight_ind"])).reset_index(name='activity_nh_ratio')


,Household_type,activity_nh_ratio
0,0.0,4.615163
1,1.0,4.225408
2,2.0,4.528072
3,3.0,5.254419
4,4.0,4.773181
5,5.0,3.973394
6,6.0,4.710616
7,7.0,6.023193


### 2.2 Active mode users

In [ ]:
# Extract unique h3_ids
h3_list_a = df.loc[df['active_mode'] == 1, 'h3_id'].unique()
id_list = df.loc[df['active_mode'] == 1, 'ID'].unique()

In [ ]:
# Extract unique h3_ids
h3_list = df.loc[:, 'h3_id'].unique()
polygons = [Polygon(h3.cells_to_geo([x])['coordinates'][0]) for x in h3_list]

# Create a GeoDataFrame from unique h3_ids
gdf_h3 = gpd.GeoDataFrame(
    h3_list,
    geometry=polygons,
    crs="EPSG:4326"
)
gdf_h3.columns= ['h3_id', 'geometry']
gdf_h3.loc[:, 'active'] = gdf_h3['h3_id'].apply(lambda x: 1 if x in h3_list_a else 0)
fig, ax = plt.subplots(figsize=(8, 6))
gdf_h3.plot(
    column="active",       # use your column for colors
    cmap="viridis",        # or "coolwarm", "plasma", etc.
    legend=True, 
    ax=ax,
    edgecolor="none"       # optional: cleaner look
)
ax.set_axis_off()
plt.show()

In [ ]:
df_bg = pd.read_csv('dbs/data_p/commuter_time_budget.csv')
df_bg.loc[:, 'active'] = df_bg['ID'].apply(lambda x: 1 if x in id_list else 0)
df_bg = df_bg.merge(df[['ID', 'weight_ind']], on='ID', how='left').dropna()
df_bg.groupby('active').apply(lambda g: np.average(g["time_hw"], weights=g["weight_ind"])).reset_index(name='time_hw')

C:\Users\yuanlia\AppData\Local\Temp\ipykernel_14408\784439981.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_bg.groupby('active').apply(lambda g: np.average(g["time_hw"], weights=g["weight_ind"])).reset_index(name='time_hw')


,active,time_hw
0,0,42.627884
1,1,33.584260


### 2.3 Travel time to leisure activities

In [ ]:
# --- filter trips where origin OR destination is ACCOM ---
df_leisure = df_trips[(df_trips["purpose_d"] == "LEISURE") & df_trips['dow'].isin(['monday', 'tuesday', 'wedsday', 'thursday', 'friday'])].copy()

# --- group by ID and compute weighted median duration ---
wm_by_id = (
    df_leisure.groupby("ID")
    .apply(lambda g: weighted_median(g["duration"].to_numpy(), g["weight_day"].to_numpy()))
    .reset_index(name="leisure_travel_time")
)

print(wm_by_id.head())

In [24]:
df.columns

Index(['ID', 'entropy_mm', 'activity_nh_ratio', 'activity_time_third',
       'total_travel_time', 'xs_total_hws', 'trip_chaining_presence', 'Gender',
       'main_mode_r', 'Age', 'Education', 'Household_type', 'weight_ind',
       'access_h', 'mode', 'ak', 'ak_cat', 'ak_log', 'codgeo', 'pt_sub',
       'entropy_mm_norm', 'active_mode', 'ah_log', 'ak_zero', 'accom_time'],
      dtype='object')

In [25]:
df_trips = df_trips.merge(df[['ID', 'weight_ind']], on='ID', how='left')
df_trips.dropna(inplace=True)
df_trips.loc[:, 'wt'] = df_trips.loc[:, 'weight_day']*df_trips.loc[:, 'weight_ind']

In [26]:
df_trips.groupby('purpose_d').apply(lambda g: np.average(g["duration"], weights=g["wt"])).reset_index(name='travel_time')

C:\Users\yuanlia\AppData\Local\Temp\ipykernel_9352\1039467619.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_trips.groupby('purpose_d').apply(lambda g: np.average(g["duration"], weights=g["wt"])).reset_index(name='travel_time')


,purpose_d,travel_time
0,ACCOM,17.249014
1,BUSINESS,30.838027
2,HEALTH,20.916253
3,LEISURE,24.609823
4,OTHER,24.905197
5,PURCHASE,16.531546
6,RETURN_HOME,25.403713
7,STUDIES,25.898399
8,WORK,32.547834


## 2.4 Correlation between activity-travel ratio vs. trip chaining complexity

In [28]:
from scipy.stats import spearmanr

# Example: columns x and y in df
corr, pval = spearmanr(df['activity_nh_ratio'], df['xs_total_hws'], nan_policy='omit')

print(f"Spearman correlation: {corr:.3f}")
print(f"P-value: {pval:.3g}")

Spearman correlation: 0.117
P-value: 5.78e-09
